# Collation automatique et export TEI-XML

Ce notebook prépare un corpus de témoins TEI, effectue la collation mot à mot avec CollateX et exporte le résultat dans un document TEI-XML. Il repose sur un pré-alignement manuel par identifiants de vers.

## Import des bibliothèques

In [ ]:
from pathlib import Path
from dataclasses import dataclass
from datetime import datetime
from typing import List, Dict, Optional, Tuple
from lxml import etree
from collatex import collate
import itertools
import json
import re
import unicodedata

## Configuration

In [ ]:
repo_root = Path.cwd()
input_dir = repo_root / "cliges8" #charger corpus ici, le notebook dans le même dossier
output_dir = repo_root / "exports"
output_dir.mkdir(parents=True, exist_ok=True)

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_path = output_dir / f"{stamp}__cliges_collation.tei.xml"

## Fonctions de normalisation et de tri

In [ ]:
LINE_ID_RE = re.compile(r"(\d+)(.*)")

def split_line_id(line_id: str) -> Tuple[int, str]:
    m = LINE_ID_RE.fullmatch(str(line_id).strip())
    if not m:
        return (10**9, str(line_id))
    return (int(m.group(1)), m.group(2))

def strip_accents(text: str) -> str:
    text = unicodedata.normalize("NFKD", text)
    return "".join(ch for ch in text if not unicodedata.combining(ch))

def normalize_token(text: str) -> str:
    text = strip_accents(text.lower())
    text = text.replace("v", "u")
    text = text.replace("j", "i")
    text = text.replace("y", "i")
    text = text.replace("z", "s")
    text = text.replace("ç", "c")
    text = text.replace("œ", "oe")
    text = text.replace("æ", "ae")
    text = re.sub(r"[^a-z]+", "", text)
    return text

## Objets de travail

In [ ]:
@dataclass
class Witness:
    siglum: str
    path: Path

    def __post_init__(self):
        self.tree = etree.parse(str(self.path))

    def line_ids(self) -> List[str]:
        return self.tree.xpath("//l/@id")

    def get_line_by_id(self, line_id: str):
        result = self.tree.xpath("//l[@id=$line_id]", line_id=str(line_id))
        return result[0] if result else None


class Line:
    add_w_milestones = etree.XML("""
<xsl:stylesheet version="1.0" xmlns:xsl="http://www.w3.org/1999/XSL/Transform">
    <xsl:output method="xml" indent="no" encoding="UTF-8" omit-xml-declaration="yes"/>

    <xsl:template match="*|@*">
        <xsl:copy>
            <xsl:apply-templates select="node() | @*"/>
        </xsl:copy>
    </xsl:template>

    <xsl:template match="/*">
        <xsl:copy>
            <xsl:apply-templates select="@*"/>
            <w/>
            <xsl:apply-templates/>
        </xsl:copy>
    </xsl:template>

    <xsl:template match="add | del | subst | abbr | expan | choice | orig | reg | pb">
        <xsl:element name="{name()}">
            <xsl:attribute name="n">start</xsl:attribute>
        </xsl:element>
        <xsl:apply-templates/>
        <xsl:element name="{name()}">
            <xsl:attribute name="n">end</xsl:attribute>
        </xsl:element>
    </xsl:template>

    <xsl:template match="note"/>

    <xsl:template match="text()">
        <xsl:call-template name="whiteSpace">
            <xsl:with-param name="input" select="translate(., '&#x0a;', ' ')"/>
        </xsl:call-template>
    </xsl:template>

    <xsl:template name="whiteSpace">
        <xsl:param name="input"/>
        <xsl:choose>
            <xsl:when test="not(contains($input, ' '))">
                <xsl:value-of select="$input"/>
            </xsl:when>
            <xsl:when test="starts-with($input, ' ')">
                <xsl:call-template name="whiteSpace">
                    <xsl:with-param name="input" select="substring($input, 2)"/>
                </xsl:call-template>
            </xsl:when>
            <xsl:otherwise>
                <xsl:value-of select="substring-before($input, ' ')"/>
                <w/>
                <xsl:call-template name="whiteSpace">
                    <xsl:with-param name="input" select="substring-after($input, ' ')"/>
                </xsl:call-template>
            </xsl:otherwise>
        </xsl:choose>
    </xsl:template>
</xsl:stylesheet>
""")

    transform_add_w = etree.XSLT(add_w_milestones)

    wrap_w = etree.XML("""
<xsl:stylesheet xmlns:xsl="http://www.w3.org/1999/XSL/Transform" version="1.0">
    <xsl:output method="xml" indent="no" omit-xml-declaration="yes"/>

    <xsl:template match="/*">
        <xsl:copy>
            <xsl:apply-templates select="w"/>
        </xsl:copy>
    </xsl:template>

    <xsl:template match="w">
        <xsl:variable name="tooFar" select="following-sibling::w[1] | following-sibling::w[1]/following::node()"/>
        <w>
            <xsl:copy-of select="following-sibling::node()[count(. | $tooFar) != count($tooFar)]"/>
        </w>
    </xsl:template>
</xsl:stylesheet>
""")

    transform_wrap_w = etree.XSLT(wrap_w)
    strip_tags_re = re.compile(r"<.*?>", re.DOTALL)

    def __init__(self, element):
        self.element = element

    @staticmethod
    def unwrap_word(word_element) -> str:
        xml = etree.tostring(word_element, encoding="unicode")
        m = re.fullmatch(r"<w>(.*)</w>", xml, flags=re.DOTALL)
        return m.group(1) if m else xml

    @classmethod
    def normalized_form(cls, raw_token: str) -> str:
        stripped = cls.strip_tags_re.sub("", raw_token)
        return normalize_token(stripped)

    def tokens(self) -> List[Dict[str, str]]:
        wrapped = Line.transform_wrap_w(Line.transform_add_w(self.element))
        tokens = []
        for token in wrapped.xpath("//w"):
            raw = self.unwrap_word(token)
            normalized = self.normalized_form(raw)
            if normalized:
                tokens.append({"t": raw, "n": normalized})
        return tokens

    def plain_text(self) -> str:
        return " ".join(token["n"] for token in self.tokens())


class WitnessSet:
    def __init__(self, witnesses: List[Witness]):
        self.witnesses = witnesses

    @classmethod
    def from_directory(cls, directory: Path):
        witnesses = []
        for path in sorted(Path(directory).glob("*.xml")):
            witnesses.append(Witness(siglum=path.stem, path=path))
        return cls(witnesses)

    def all_line_ids(self) -> List[str]:
        ids = set(itertools.chain.from_iterable(w.line_ids() for w in self.witnesses))
        return sorted(ids, key=split_line_id)

    def get_lines_by_id(self, line_id: str):
        lines = []
        for witness in self.witnesses:
            line = witness.get_line_by_id(line_id)
            if line is not None:
                lines.append((witness.siglum, line))
        return lines

    def generate_collatex_input(self, line_id: str) -> Dict:
        witnesses = []
        for siglum, line in self.get_lines_by_id(line_id):
            witnesses.append({
                "id": siglum,
                "tokens": Line(line).tokens()
            })
        return {"witnesses": witnesses}

## Chargement du corpus

In [ ]:
witset = WitnessSet.from_directory(input_dir)
all_line_ids = witset.all_line_ids()

len(witset.witnesses), len(all_line_ids)

## Vérification sur un vers

In [ ]:
sample_line_id = all_line_ids[0]
sample_input = witset.generate_collatex_input(sample_line_id)

sample_line_id, sample_input

## Collation en XML pour un vers

In [ ]:
sample_xml = collate(sample_input, output="xml", segmentation=False)
sample_xml

## Fonctions d'export TEI

In [ ]:
TEI_NS = "http://www.tei-c.org/ns/1.0"
XML_NS = "http://www.w3.org/XML/1998/namespace"
NSMAP = {None: TEI_NS}

def collate_line_to_xml(witset: WitnessSet, line_id: str) -> Optional[str]:
    data = witset.generate_collatex_input(line_id)
    if not data["witnesses"]:
        return None
    return collate(data, output="xml", segmentation=False)

def build_tei_collation(witset: WitnessSet):
    root = etree.Element(f"{{{TEI_NS}}}TEI", nsmap=NSMAP)

    teiHeader = etree.SubElement(root, f"{{{TEI_NS}}}teiHeader")
    fileDesc = etree.SubElement(teiHeader, f"{{{TEI_NS}}}fileDesc")
    titleStmt = etree.SubElement(fileDesc, f"{{{TEI_NS}}}titleStmt")
    title = etree.SubElement(titleStmt, f"{{{TEI_NS}}}title")
    title.text = "Collation automatique de Cligés"

    publicationStmt = etree.SubElement(fileDesc, f"{{{TEI_NS}}}publicationStmt")
    p_pub = etree.SubElement(publicationStmt, f"{{{TEI_NS}}}p")
    p_pub.text = "Export généré automatiquement."

    sourceDesc = etree.SubElement(fileDesc, f"{{{TEI_NS}}}sourceDesc")
    p_source = etree.SubElement(sourceDesc, f"{{{TEI_NS}}}p")
    p_source.text = "Collation réalisée à partir de témoins TEI préalignés manuellement."

    encodingDesc = etree.SubElement(teiHeader, f"{{{TEI_NS}}}encodingDesc")
    variantEncoding = etree.SubElement(encodingDesc, f"{{{TEI_NS}}}variantEncoding")
    variantEncoding.set("method", "parallel-segmentation")
    variantEncoding.set("location", "internal")

    profileDesc = etree.SubElement(teiHeader, f"{{{TEI_NS}}}profileDesc")
    listWit = etree.SubElement(profileDesc, f"{{{TEI_NS}}}listWit")
    for witness in witset.witnesses:
        wit = etree.SubElement(listWit, f"{{{TEI_NS}}}witness")
        wit.set(f"{{{XML_NS}}}id", witness.siglum)
        wit.text = witness.siglum

    text = etree.SubElement(root, f"{{{TEI_NS}}}text")
    body = etree.SubElement(text, f"{{{TEI_NS}}}body")
    div = etree.SubElement(body, f"{{{TEI_NS}}}div")
    div.set("type", "collation")

    for line_id in witset.all_line_ids():
        collation_xml = collate_line_to_xml(witset, line_id)
        if not collation_xml:
            continue

        line_container = etree.SubElement(div, f"{{{TEI_NS}}}l")
        line_container.set("n", str(line_id))

        wrapped = etree.fromstring(f"<wrapper>{collation_xml}</wrapper>")
        for child in wrapped:
            line_container.append(child)

    return root

def write_tei_tree(tree, path: Path):
    etree.ElementTree(tree).write(
        str(path),
        encoding="UTF-8",
        xml_declaration=True,
        pretty_print=True
    )

## Génération du document TEI complet

In [ ]:
tei_root = build_tei_collation(witset)
write_tei_tree(tei_root, output_path)

output_path

## Aperçu du résultat

In [ ]:
print(output_path.read_text(encoding="utf-8")[:4000])